# Laboratorio — Robot de entregas en un almacén - **Jerónimo Zapata Cuadros**

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [28]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0,0)
        self.walls = {(0,3),(1,1),(2,4),(4,2)}
        self.slippery_states = {(1,2),(2,1),(3,3)}

        self.terminal_states = {
            # (row, col): reward
            (0,5): 10,
            (2,2): 2,
            (3,5): -10

        }

        self.danger_states = {
            # (row, col): -3
            (1,4): -3,
            (4,1): -3
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row,col = state
        inside_grid = (
            0 <= row < self.height
            and
            0 <= col < self.width
        )

        not_wall = state not in self.walls

        return inside_grid and not_wall
        

    def states(self):
        return [
            (row,col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row,col))
        ]

    def is_terminal(self, state):
       return state in self.terminal_states

    def get_reward(self, state):

        #Primero revisamos si es terminal
        if self.is_terminal(state):
            return self.terminal_states[state]

        #Después revisamos peligros no terminales
        if state in self.danger_states:
            return self.danger_states[state]

        #Cualquier otro estado transitable
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Los estados terminales no tienen acciones posteriores
        if self.is_terminal(state):
            return [(state,1.0)]

        # Se determinan las probabilidades segun el piso
        if state in self.slippery_states:
            p_main = 0.60
            p_left = 0.20
            p_right = 0.20
        else:
            p_main = 0.90
            p_left = 0.05
            p_right = 0.05

        # 2. Determinar las direcciones de desviación
  

        dr, dc = action

        # Dirección perpendicular izquierda
        left_action = (-dc, dr)

        # Dirección perpendicular derecha
        right_action = (dc, -dr)

   
        # 3. Función para realizar un movimiento
      

        def next_state(state, direction):

            row, col = state
            dr, dc = direction

            candidate = (
                row + dr,
                col + dc
            )

            # Si es válido, nos movemos
            if self.is_valid_state(candidate):
                return candidate

            # Si hay pared o borde, permanecemos
            return state

  
        # 4. Calcular los tres posibles resultados
      

        main_state = next_state(state, action)
        left_state = next_state(state, left_action)
        right_state = next_state(state, right_action)

    
        # 5. Agrupar estados repetidos
     

        probabilities = {}

        for next_s, probability in [
            (main_state, p_main),
            (left_state, p_left),
            (right_state, p_right)
        ]:

            if next_s in probabilities:
                probabilities[next_s] += probability
            else:
                probabilities[next_s] = probability

        return list(probabilities.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [29]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [30]:
def expected_next_value(grid, state, action, V):
    # TODO:
    # sum_{s'} T(s,a,s') V(s')
    expected_value = 0.0
    transitions = grid.get_transition_probs(state,action)

    for next_state, probability in transitions:
        expected_value += probability * V[next_state]

    return expected_value


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # TODO
    V = {
        state: 0.0
        for state in grid.states()
    }

    for iteration in range(max_iter):
        V_new = V.copy()
        delta = 0.0
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
                continue

            #Calculamos el valor esperado de cada acción
            action_values = []
            for action in grid.actions:
                value = expected_next_value(
                    grid,
                    state,
                    action,
                    V
                )
                action_values.append(value)

            #Elegimos la mejor acción
            best_action_value = max(action_values)
            # Ecuación de Bellman
            V_new[state] = (
                grid.get_reward(state) + grid.gamma * best_action_value
            )

            #Error máximo de esta iteración
            delta = max(
                delta,
                abs(V_new[state] - V[state])
            )

        V = V_new
        if delta < threshold:
            n_iter = iteration + 1
            return V, n_iter

    return V, max_iter


def extract_policy(grid, V):
    # TODO:
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            policy[state] = None
            continue

        best_action = None
        best_value = -np.inf

        for action in grid.actions:
            value = expected_next_value(
                grid,
                state,
                action,
                V
            )
            if value > best_value:
                best_value = value
                best_action = action
        policy[state] = best_action
        
    return policy

    



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [31]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # TODO
    V = {
        state : 0.0
        for state in grid.states()
    }

    for iteration in range(max_iter):
        # V_new representa V_{k+1}
        V_new = V.copy()
        delta = 0.0
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
                continue

            action = policy[state]
            expected_value = expected_next_value(
                grid,
                state,
                action,
                V
            )

            #Ecuación de Policy Evaluation
            V_new[state] = (
                grid.get_reward(state) + grid.gamma * expected_value
            )

            delta = max(
                delta,
                abs(V_new[state] - V[state])
            )

        V = V_new
        if delta < threshold:
            break

    return V

def policy_improvement(grid, V):
    # TODO
    policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            policy[state] = None
            continue

        best_action = None
        best_value = -np.inf

        for action in grid.actions:
            expected_value = expected_next_value(
                grid,
                state,
                action,
                V
            )

            if expected_value > best_value:
                best_value = expected_value
                best_action = action

        policy[state] = best_action

    return policy



def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # TODO:
    # 1. política inicial arbitraria
    policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            policy[state] = None
        else:
            policy[state] = grid.actions[0]

    history = [policy.copy()]

    # 2. evaluación y mejora
    for iteration in range(max_iter):
        #Policy Evaluation
        V = policy_evaluation(
            grid,
            policy,
            threshold=threshold
        )

        #Policy Improvement
        new_policy = policy_improvement(
            grid,
            V
        )

    # 3. repetir hasta estabilidad
        policy_stable = True

        for state in grid.states():
            if policy[state] != new_policy[state]:
                policy_stable = False
                break

        if policy_stable:
            print(f"Política estable en" f"{iteration + 1} iteraciones.")
            policy = new_policy
            break
        policy = new_policy

    return policy, V, history

    



## Parte 4 — Visualización y comparación


In [32]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [33]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Política estable en4 iteraciones.

=== POLICY ITERATION ===
Historia: [{(0, 0): (-1, 0), (0, 1): (-1, 0), (0, 2): (-1, 0), (0, 4): (-1, 0), (0, 5): None, (1, 0): (-1, 0), (1, 2): (-1, 0), (1, 3): (-1, 0), (1, 4): (-1, 0), (1, 5): (-1, 0), (2, 0): (-1, 0), (2, 1): (-1, 0), (2, 2): None, (2, 3): (-1, 0), (2, 5): (-1, 0), (3, 0): (-1, 0), (3, 1): (-1, 0), (3, 2): (-1, 0), (3, 3): (-1, 0), (3, 4): (-1, 0), (3, 5): None, (4, 0): (-1, 0), (4, 1): (-1, 0), (4, 3): (-1, 0), (4, 4


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

Desde Start = (0,0), el robot busca la entrega de +10. Esto se puede ver en la política obtenida, porque desde (0,0) la primera acción es  →  y después sigue avanzando hacia la derecha hasta llegar a la zona de entrega. La estación de carga (2,2) también tiene una recompensa positiva de +2, pero la política óptima no la escoge desde el inicio porque, teniendo en cuenta las recompensas, los costos por paso y el factor de descuenta, la entrega termina sienda una opción más conveniente.
Además, el valor del estado inicial es aproximadamente: V(0,0) = -2.575. Esto no significa que la entrega sea mala, sino que llegar hasta ella implica pasar por varios esatdos que tienen costo -1 y además existe incertidumbre en los movimientos.

2. ¿Por qué una recompensa menor podría ser óptima?

Una recompensa menor podría ser óptima porque el robot no solamente tiene en cuenta la recompensa final, sino también todo lo que puede pasar durante el camino. Por ejemplo, la carga da +2, mientras que la entrega da +10. A primera visto uno pensaría que siempre hay que ir por el +10, pero el robot también tiene que considerar los costos por cada paso, las probabilidades de desviarse y las recompensas negativas.
Entonces podría darse el caso de que llegar al +2 sea mucho más seguro o requiera menos pasos que llegar al +10. En ese caso, aunque+2 sea menor, el valor esperado de llegar ahí podría ser mejor.

3. ¿En qué estados el piso resbaloso cambia la decisión?

Los estados resbalosos son (1,2), (2,1) y (3,3). En los resultados se puede notar que especialmente en la zona alredor de estos estados, la política no simplemente busca el camino más directo, sino que toma en cuenta las postibles desviaciones. Por ejemplo, en (1,2) la política obtenida es ↓, mientras que en (2,1) es →.  Esto muestra que la política está considerando la dinámica probabilística del piso y no solamente la posición de la recompensa.

4. ¿Qué papel cumple el costo por paso `-1`?

El costo de -1 hace que el robot tenga un incentivo para no dar vueltas ni tomar caminos innecesariamente largos, así, el robot buscará llegar a una buena recompensa tratando de gastar la menor cantidad posible de pases y evitando situaciones peligrosas.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

Porque las probabilidades dependen del tipo de piso donde se encuentra el robot. Por ejemplo, hacer RIGHT desde un piso normal no tiene las mismas probabilidades de hacer RIGHT desde (1,2), porque (1,2) es resbaloso.
Además, si el movimiento intetnta salir del mapa o entrar en una estantería, el robot se queda en el mismo estado. Por eso las probabilidades también dependen de qué haya alrededor del estado actual.


### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

Mi predicción sería que el robot va a estar más dispuesto a tomar caminos largos para intentar llegar a la entrega de +10, porque cada paso ahora cuesta solamente -0.1 en vez de -1.

La predicción fue correcta. Al reducir el costo por paso, el agente está más dispuesto a tomar caminos largos porque la penalización acumulada por desplazarse es menor. Por eso resulta más atractivo intentar alcanzar la recompensa de +10 en lugar de conformarse con la recompensa cercana de +2.

In [34]:
grid_A = WarehouseMDP()

# Cambiamos únicamente el costo por paso
grid_A.living_reward = -0.1

# Value Iteration
V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)

print("EXPERIMENTO A: living_reward = -0.1")
print("Iteraciones:", n_A)

print("\nValores:")
print_values(grid_A, V_A)

print("\nPolítica:")
print_policy(grid_A, pi_A)

print("\nAcción desde START:", ARROWS[pi_A[grid_A.start]])

EXPERIMENTO A: living_reward = -0.1
Iteraciones: 26

Valores:
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Acción desde START: →


### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

**Interpretación:** aunque el aumento de la incertidumbre modifica los valores y algunas decisiones intermedias, no es suficiente para cambiar la decisión inicial desde START. El agente sigue considerando que intentar llegar a +10 es la mejor estrategia

In [35]:
class SlipperyWarehouseMDP(WarehouseMDP):

    def get_transition_probs(self, state, action):

        # Si es un estado terminal, no hay movimientos
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Probabilidades para piso normal
        if state in self.slippery_states:

            # EXPERIMENTO B
            intended_prob = 0.40
            side_prob = 0.30

        else:

            intended_prob = 0.90
            side_prob = 0.05

        def move(state, action):

            r, c = state
            dr, dc = action

            next_state = (r + dr, c + dc)

            if not self.is_valid_state(next_state):
                return state

            return next_state

        action_index = self.actions.index(action)

        # Desviaciones laterales
        left_index = (action_index - 1) % 4
        right_index = (action_index + 1) % 4

        left_action = self.actions[left_index]
        right_action = self.actions[right_index]

        transitions = [
            (move(state, action), intended_prob),
            (move(state, left_action), side_prob),
            (move(state, right_action), side_prob)
        ]

        probs = {}

        for next_state, probability in transitions:

            probs[next_state] = (
                probs.get(next_state, 0)
                + probability
            )

        return list(probs.items())



grid_B = SlipperyWarehouseMDP()

V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)

print("EXPERIMENTO B: piso resbaloso 0.40 / 0.30 / 0.30")
print("Iteraciones:", n_B)

print("\nValores:")
print_values(grid_B, V_B)

print("\nPolítica:")
print_policy(grid_B, pi_B)

print("\nAcción desde START:", ARROWS[pi_B[grid_B.start]])

EXPERIMENTO B: piso resbaloso 0.40 / 0.30 / 0.30
Iteraciones: 23

Valores:
 -2.874 |  -1.994 |  -0.957 |   WALL   |  +7.802 | +10.000
 -2.902 |   WALL   |  +0.217 |  +2.218 |  +3.838 |  +7.670
 -2.028 |  -0.995 |  +2.000 |  +0.761 |   WALL   |  +4.987
 -1.698 |  -0.654 |  +0.576 |  -1.545 |  -2.853 | -10.000
 -2.670 |  -3.879 |   WALL   |  -2.516 |  -3.375 |  -4.103

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ←  |  →  |  ↑  |  ↑ 
 →  |  ↑  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↓  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Acción desde START: →


### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

**Interpretación:** aumentar gamma hace que el agente sea más paciente y valore más las recompensas futuras. En este caso, esto favorece la estrategia de intentar alcanzar la recompensa +10. Sin embargo, como con gamma=0.9 ya era óptimo dirigirse hacia +10, aumentar gamma no produce un cambio visible en la acción desde START; simplemente hace que esa recompensa futura tenga todavía más peso en los valores calculados.

In [36]:
# EXPERIMENTO C — MÁS PACIENCIA

grid_C = WarehouseMDP()

grid_C.gamma = 0.99

# Value Iteration
V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)

print("EXPERIMENTO C: gamma = 0.99")
print("Iteraciones:", n_C)

print("\nValores:")
print_values(grid_C, V_C)

print("\nPolítica:")
print_policy(grid_C, pi_C)

print("\nAcción desde START:", ARROWS[pi_C[grid_C.start]])

EXPERIMENTO C: gamma = 0.99
Iteraciones: 24

Valores:
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Acción desde START: →


### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

#### **Interpretación de los resultados**
Al variar living_reward, se encontró que la política óptima desde START cambia aproximadamente entre -108.864 y -108.863. Para valores más negativos, la acción óptima desde START es ↓, lo que corresponde a dirigirse hacia la recompensa +2. Para valores mayores, la acción cambia a →, que corresponde a intentar llegar a la recompensa +10. Por tanto, el valor crítico de living_reward es aproximadamente -108.864.

Esto tiene una explicación, ya que a medida que living_reward se vuelve más negativo, el agente penaliza cada paso que da. Cuando la penalización es suficientemente grande, prefiere tomar la recompensa cercana +2 en lugar de recorrer una ruta más larga para intentar obtener +10. 

In [37]:
living_rewards = np.arange(-108.9, -108.8, 0.001)

previous_action = None

for reward in living_rewards:

    grid_bonus = WarehouseMDP()
    grid_bonus.living_reward = reward

    V_bonus, n_bonus = value_iteration(grid_bonus)
    pi_bonus = extract_policy(grid_bonus, V_bonus)

    current_action = pi_bonus[grid_bonus.start]

    if (
        previous_action is not None
        and current_action != previous_action
    ):

        print("Cambio de política detectado.")
        print("Aproximadamente entre:")
        print(f"{reward - 0.001:.3f} y {reward:.3f}")

        print("Antes:", ARROWS[previous_action])
        print("Después:", ARROWS[current_action])

        break

    previous_action = current_action

Cambio de política detectado.
Aproximadamente entre:
-108.864 y -108.863
Antes: ↓
Después: →
